# General migration

Copy every collection (schema, objects, vectors, tenants) from one Weaviate instance to another: open source (Docker, Kubernetes), Weaviate on DigitalOcean, Weaviate Cloud, or anything else. Objects are re-inserted with their original UUIDs and vectors, so nothing is re-vectorized.

The same logic is available as a command-line script: `general_migration.py`.

Notes:
- Both instances must expose gRPC (default port 50051) as well as HTTP.
- Vectorizer modules used by the source collections must be enabled on the target.
- Cross-references are not migrated.
- Shard count and replication factor come from the target's defaults.
- Only HOT (active) tenants can be read. COLD or offloaded tenants are skipped; activate them first.
- Other settings the source leaves unset also take the target's defaults, e.g. Weaviate Cloud compresses uncompressed vectors with RQ.
- Existing collections and tenants on the target are reused and objects with the same UUID are overwritten, so it is safe to re-run.


In [ ]:
%pip install -U weaviate-client

In [ ]:
# --- CONFIGURATION ---
# Self-hosted (Docker, Kubernetes, DigitalOcean, ...): host without http:// or https://, HTTP port
# (default 8080), gRPC port (default 50051), secure=True for HTTPS/TLS, api_key=None for anonymous access.
# Weaviate Cloud: host="abc.c0.europe-west3.gcp.weaviate.cloud", grpc_host="grpc-abc.c0.europe-west3.gcp.weaviate.cloud",
# http_port=443, grpc_port=443, secure=True (or replace connect() with
# weaviate.connect_to_weaviate_cloud(cluster_url=..., auth_credentials=Auth.api_key(...))).
SOURCE = {"host": "localhost", "http_port": 8080, "grpc_host": None, "grpc_port": 50051, "secure": False, "api_key": None}
TARGET = {"host": "<TARGET-HOST>", "http_port": 443, "grpc_host": None, "grpc_port": 443, "secure": True, "api_key": "<TARGET-API-KEY>"}

COLLECTIONS = None  # e.g. ["Articles", "Products"]; None copies every collection
HEADERS = {}  # vectorizer API keys, needed only if the target has to vectorize something, e.g. {"X-OpenAI-Api-Key": "sk-..."}

In [ ]:
import logging
import sys
import httpx
import weaviate
from weaviate.classes.init import AdditionalConfig, Auth, Timeout
from weaviate.classes.tenants import Tenant, TenantActivityStatus
from weaviate.client import WeaviateClient
from weaviate.collections import Collection

HOT_STATUSES = (TenantActivityStatus.ACTIVE, TenantActivityStatus.HOT)

logger = logging.getLogger("migration")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-5s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)


def connect(cfg: dict) -> WeaviateClient:
    return weaviate.connect_to_custom(
        http_host=cfg["host"],
        http_port=cfg["http_port"],
        http_secure=cfg["secure"],
        grpc_host=cfg.get("grpc_host") or cfg["host"],
        grpc_port=cfg["grpc_port"],
        grpc_secure=cfg["secure"],
        headers=HEADERS,
        auth_credentials=Auth.api_key(cfg["api_key"]) if cfg.get("api_key") else None,
        additional_config=AdditionalConfig(timeout=Timeout(init=60, query=240, insert=480)),
    )


def fetch_schema(cfg: dict) -> dict:
    """Return {collection name: raw config} straight from the REST API.

    The client's export_config().to_dict() loses some settings (multi-vector indexes come back
    disabled), so the schema is copied exactly as the server reports it.
    """
    scheme = "https" if cfg["secure"] else "http"
    headers = {"Authorization": f"Bearer {cfg['api_key']}"} if cfg.get("api_key") else {}
    response = httpx.get(f"{scheme}://{cfg['host']}:{cfg['http_port']}/v1/schema", headers=headers, timeout=60)
    response.raise_for_status()
    return {c["class"]: c for c in response.json().get("classes") or []}


def server_version(client: WeaviateClient) -> tuple:
    version = client.get_meta()["version"].split("-")[0]
    return tuple(int(part) for part in version.split(".")[:2])


def copy_schema(target: WeaviateClient, config: dict) -> None:
    name = config["class"]
    if target.collections.exists(name):
        logger.info("%s: already exists on the target, reusing it", name)
        return

    # Shard layout and replication factor depend on the cluster size, so the target uses its own
    # defaults (some deployments enforce a minimum replication factor).
    config = dict(config)
    config.pop("shardingConfig", None)
    replication = dict(config.get("replicationConfig") or {})
    replication.pop("factor", None)
    config["replicationConfig"] = replication

    target.collections.create_from_dict(config)
    logger.info("%s: created on the target", name)


def copy_objects(source_col: Collection, target_col: Collection, label: str,
                 legacy_vector: bool, server_side_batching: bool) -> tuple:
    """Copy all objects; return (copied, failures) with one {migration, uuid, error} dict per failed object."""
    count = 0
    # Server-side batching (Weaviate >= 1.36) lets the server control the pace; older targets use dynamic batching.
    batching = target_col.batch.stream() if server_side_batching else target_col.batch.dynamic()
    with batching as batch:
        for obj in source_col.iterator(include_vector=True):
            batch.add_object(
                properties=obj.properties,
                uuid=obj.uuid,
                # Collections without named vectors take a single vector, not a dict.
                vector=obj.vector.get("default") if legacy_vector else obj.vector,
            )
            count += 1
            if count % 10_000 == 0:
                logger.info("%s: %d objects sent", label, count)

    failures = [{"migration": label, "uuid": str(f.object_.uuid), "error": f.message}
                for f in target_col.batch.failed_objects]
    if failures:
        logger.error("%s: %d of %d objects failed, first error: %s", label, len(failures), count, failures[0]["error"])
    logger.info("%s: %d objects copied", label, count - len(failures))
    return count - len(failures), failures


def migrate_collection(source: WeaviateClient, target: WeaviateClient, config: dict,
                       server_side_batching: bool) -> dict:
    name = config["class"]
    copy_schema(target, config)
    legacy_vector = not config.get("vectorConfig")
    source_col = source.collections.use(name)
    target_col = target.collections.use(name)
    result = {"copied": 0, "failures": [], "skipped_tenants": []}

    if not (config.get("multiTenancyConfig") or {}).get("enabled"):
        result["copied"], result["failures"] = copy_objects(
            source_col, target_col, name, legacy_vector, server_side_batching)
        return result

    tenants = source_col.tenants.get()
    hot = [t for t, info in tenants.items() if info.activity_status in HOT_STATUSES]
    result["skipped_tenants"] = [t for t in tenants if t not in hot]
    for tenant in result["skipped_tenants"]:
        logger.warning("%s/%s: tenant is %s, skipped (only HOT tenants can be migrated)",
                       name, tenant, tenants[tenant].activity_status.value)

    existing = target_col.tenants.get()
    new_tenants = [Tenant(name=t) for t in hot if t not in existing]
    if new_tenants:
        target_col.tenants.create(new_tenants)
    logger.info("%s: %d tenants to copy", name, len(hot))

    for tenant in hot:
        copied, failures = copy_objects(source_col.with_tenant(tenant), target_col.with_tenant(tenant),
                                        f"{name}/{tenant}", legacy_vector, server_side_batching)
        result["copied"] += copied
        result["failures"] += failures
    return result

In [ ]:
schema = fetch_schema(SOURCE)
names = COLLECTIONS or list(schema)
missing = [n for n in names if n not in schema]
assert not missing, f"not found on the source: {missing}"

with connect(SOURCE) as source, connect(TARGET) as target:
    server_side_batching = server_version(target) >= (1, 36)
    logger.info("source %s, target %s, %d collections to copy",
                source.get_meta()["version"], target.get_meta()["version"], len(names))

    results = {}
    for name in names:
        try:
            results[name] = migrate_collection(source, target, schema[name], server_side_batching)
        except Exception as e:
            logger.error("%s: migration failed: %s", name, e)
            results[name] = {"copied": 0, "failures": None, "skipped_tenants": []}

logger.info("summary:")
for name, r in results.items():
    if r["failures"] is None:
        logger.info("  %s: FAILED, see the error above", name)
        continue
    skipped = f", skipped tenants: {', '.join(r['skipped_tenants'])}" if r["skipped_tenants"] else ""
    logger.info("  %s: %d copied, %d failed%s", name, r["copied"], len(r["failures"]), skipped)

failures = [f for r in results.values() for f in r["failures"] or []]
if failures:
    logger.error("%d failed objects (fix the cause, then re-insert them from the source by UUID):", len(failures))
    for f in failures:
        print(f"  {f['migration']} | {f['uuid']} | {f['error']}")